# RBL-4 Full Experiment Analysis

**Study:** Evaluating first-attempt zero-shot GPT-4o unit-test generation on Java and Python units with CC 5–15  
**Protocol:** `team-synthesis/proposal.md` (approved)  
**Analysis script:** `evaluation/integrated_statistics.py`  
**Generated:** 2026-07-22  
**Author (Statistics Lead):** Quân  

---

## Dataset summary

| Property | Value |
|---|---|
| Total units | 50 (25 Java + 25 Python) |
| Manifest | `experiment/dataset-manifest.csv` (frozen) |
| CC range | 5–15 (Lizard 1.23.0) |
| LLM | gpt-4o-mini (codex-session; GPT-4o-2024-08-06 API rejected with 429) |
| Randoop | 4.3.4, seed=20260621, 60 s/unit, JDK 17 |
| Statistical plan | Registered before data collection; no amendments |

---

## Statistical plan (proposal §6)

| RQ | Test | Multiplicity | α |
|---|---|---|---|
| RQ1 Java/Python | Exact sign test, right-tailed | Holm, 2 tests | 0.05 |
| RQ2 Java | Paired Wilcoxon, right-tailed | Single test | 0.05 |
| RQ3 | Spearman, left-tailed | Holm, 3 testable (1 NOT_TESTABLE) | 0.05 |

In [ ]:
# Cell 1 — Imports and paths
import csv
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import binomtest, spearmanr, wilcoxon

ROOT = Path().resolve().parent  # workspace root when running from results/
# If running from workspace root, adjust:
if not (ROOT / 'results').exists():
    ROOT = Path().resolve()

RESULTS = ROOT / 'results'
print(f'ROOT = {ROOT}')
print(f'RESULTS = {RESULTS}')

In [ ]:
# Cell 2 — Load all CSV files
java_cov = pd.read_csv(RESULTS / 'java_coverage.csv')
java_mut = pd.read_csv(RESULTS / 'java_mutation.csv')
java_exec = pd.read_csv(RESULTS / 'java_executability.csv')
randoop = pd.read_csv(RESULTS / 'randoop_java_results.csv')
py_cov = pd.read_csv(RESULTS / 'python_coverage.csv')
py_mut = pd.read_csv(RESULTS / 'python_mutation.csv')
py_exec = pd.read_csv(RESULTS / 'python_executability.csv')
summary = pd.read_csv(RESULTS / 'summary.csv')

print('Files loaded:')
for name, df in [('java_coverage', java_cov), ('java_mutation', java_mut),
                 ('randoop', randoop), ('python_coverage', py_cov),
                 ('python_mutation', py_mut), ('summary', summary)]:
    print(f'  {name}: {len(df)} rows')

In [ ]:
# Cell 3 — Executability overview
java_exec_ok = (java_exec['status'] == 'EXECUTABLE').sum()
py_exec_ok_count = py_cov['executability'].eq('EXECUTABLE').sum()

print('=== Executability ===')
print(f'Java:   {java_exec_ok}/25 EXECUTABLE ({100*java_exec_ok/25:.1f}%)')
print(f'Python: {py_exec_ok_count}/25 EXECUTABLE ({100*py_exec_ok_count/25:.1f}%)')

# Non-executable Java
java_fail = java_exec[java_exec['status'] != 'EXECUTABLE'][['unit_id', 'status', 'error_type']]
print('\nNon-executable Java units:')
print(java_fail.to_string(index=False))

# Non-executable Python
py_fail = py_cov[py_cov['executability'] != 'EXECUTABLE'][['unit_id', 'executability', 'notes']]
print('\nNon-executable Python units:')
print(py_fail.to_string(index=False))

In [ ]:
# Cell 4 — Descriptive statistics: branch coverage
java_bc = pd.to_numeric(java_cov['branch_coverage'], errors='coerce').dropna()
py_bc = pd.to_numeric(py_cov['branch_coverage'], errors='coerce').dropna()

print('=== Branch Coverage (%) ===')
print(f'Java  : n={len(java_bc)}, median={java_bc.median():.2f}%, '
      f'IQR=[{java_bc.quantile(0.25):.2f}, {java_bc.quantile(0.75):.2f}], '
      f'min={java_bc.min():.2f}, max={java_bc.max():.2f}')
print(f'Python: n={len(py_bc)}, median={py_bc.median():.2f}%, '
      f'IQR=[{py_bc.quantile(0.25):.2f}, {py_bc.quantile(0.75):.2f}], '
      f'min={py_bc.min():.2f}, max={py_bc.max():.2f}')

In [ ]:
# Cell 5 — Descriptive statistics: mutation score
java_ms = pd.to_numeric(java_mut['mutation_score'], errors='coerce').dropna()
py_ms = pd.to_numeric(py_mut['mutation_score'], errors='coerce').dropna()  # expected: 0 rows
randoop_ms = pd.to_numeric(randoop['randoop_mutation_score'], errors='coerce').dropna()

print('=== Mutation Score (%) ===')
print(f'Java GPT  : n={len(java_ms)}, median={java_ms.median():.2f}%, '
      f'IQR=[{java_ms.quantile(0.25):.2f}, {java_ms.quantile(0.75):.2f}]')
print(f'Java Randoop: n={len(randoop_ms)}, median={randoop_ms.median():.2f}%, '
      f'IQR=[{randoop_ms.quantile(0.25):.2f}, {randoop_ms.quantile(0.75):.2f}]')
print(f'Python GPT: n={len(py_ms)} (all NA — pytest-mutagen generated 0 mutants for all 25 functions)')

In [ ]:
# Cell 6 — RQ1: Exact sign test vs 75% threshold
THRESHOLD = 75.0

def sign_test_report(values, lang, threshold=THRESHOLD):
    above = (values > threshold).sum()
    below = (values < threshold).sum()
    ties  = (values == threshold).sum()
    n_nontie = above + below
    if n_nontie == 0:
        return dict(lang=lang, n=len(values), above=0, below=0, ties=0,
                    raw_p=1.0, note='all ties')
    result = binomtest(int(above), int(n_nontie), 0.5, alternative='greater')
    return dict(lang=lang, n=int(len(values)), above=int(above),
                below=int(below), ties=int(ties), raw_p=float(result.pvalue))

r_java   = sign_test_report(java_bc, 'Java')
r_python = sign_test_report(py_bc,   'Python')

# Holm correction
raw_ps = [r_java['raw_p'], r_python['raw_p']]
order  = sorted(range(2), key=lambda i: raw_ps[i])
adj    = [0.0, 0.0]
running = 0.0
for rank, idx in enumerate(order):
    adj[idx] = min(raw_ps[idx] * (2 - rank), 1.0)
    running = max(running, adj[idx])
    adj[idx] = running

for i, (r, adj_p) in enumerate(zip([r_java, r_python], adj)):
    decision = 'reject H0' if adj_p < 0.05 else 'fail to reject H0'
    print(f"RQ1 {r['lang']:6s}: n={r['n']}, above={r['above']}, below={r['below']}, ties={r['ties']}")
    print(f"         raw_p={r['raw_p']:.6g}, holm_adj_p={adj_p:.6g} → {decision}")
    print()

In [ ]:
# Cell 7 — RQ2: GPT-4o vs Randoop mutation score (Java only)
rq2_raw = pd.read_csv(RESULTS / 'java_rq2_analysis.csv').iloc[0]

print('=== RQ2: Paired mutation score GPT-4o vs Randoop (Java) ===')
print(f'Paired n   = {rq2_raw["paired_n"]} (missing = {rq2_raw["missing_n"]})')
print(f'GPT median = {float(rq2_raw["gpt_median"]):.2f}%')
print(f'Randoop median = {float(rq2_raw["randoop_median"]):.2f}%')
print(f'Median diff (GPT - Randoop) = {float(rq2_raw["median_difference"]):.2f} pp')
print(f'IQR of diff = [{float(rq2_raw["q1_difference"]):.2f}, {float(rq2_raw["q3_difference"]):.2f}]')
print(f'Bootstrap 95% CI = [{float(rq2_raw["bootstrap_ci_low"]):.2f}, {float(rq2_raw["bootstrap_ci_high"]):.2f}]')
print(f'Wilcoxon W = {float(rq2_raw["wilcoxon_statistic"]):.3f}, p = {float(rq2_raw["wilcoxon_p_value"]):.6g}')
print(f'Sign test: pos={rq2_raw["positive_pairs"]}, neg={rq2_raw["negative_pairs"]}, ties={rq2_raw["ties"]}, p={float(rq2_raw["sign_test_p_value"]):.6g}')
rq2_decision = 'reject H0' if float(rq2_raw['wilcoxon_p_value']) < 0.05 else 'fail to reject H0'
print(f'Decision: {rq2_decision}')

In [ ]:
# Cell 8 — RQ3: Spearman correlation CC vs test quality

def spearman_report(cc_series, quality_series, label):
    paired = pd.DataFrame({'cc': cc_series, 'q': quality_series}).dropna()
    if len(paired) < 3:
        return dict(label=label, n=len(paired), rho='NA', raw_p='NA', decision='NOT_TESTABLE')
    rho, p = spearmanr(paired['cc'], paired['q'], alternative='less')
    return dict(label=label, n=len(paired), rho=float(rho), raw_p=float(p))

java_cc  = pd.to_numeric(java_cov['cc'], errors='coerce')
java_bc2 = pd.to_numeric(java_cov['branch_coverage'], errors='coerce')
java_ms2 = pd.to_numeric(java_mut['mutation_score'], errors='coerce')
py_cc    = pd.to_numeric(py_cov['cc'], errors='coerce')
py_bc2   = pd.to_numeric(py_cov['branch_coverage'], errors='coerce')
py_ms2   = pd.to_numeric(py_mut['mutation_score'], errors='coerce')

rq3_results = [
    spearman_report(java_cc, java_bc2, 'Java × branch_coverage'),
    spearman_report(java_cc, java_ms2, 'Java × mutation_score'),
    spearman_report(py_cc,   py_bc2,   'Python × branch_coverage'),
    spearman_report(py_cc,   py_ms2,   'Python × mutation_score'),
]

# Holm on testable pairs only
testable = [(i, r) for i, r in enumerate(rq3_results) if r['decision'] != 'NOT_TESTABLE' and r['raw_p'] != 'NA']
if testable:
    t_order = sorted(testable, key=lambda x: x[1]['raw_p'])
    n_t = len(testable)
    running = 0.0
    for rank, (orig_idx, r) in enumerate(t_order):
        adj_p = min(r['raw_p'] * (n_t - rank), 1.0)
        running = max(running, adj_p)
        rq3_results[orig_idx]['adj_p'] = running
        rq3_results[orig_idx]['decision'] = (
            'reject H0' if running < 0.05 and rq3_results[orig_idx]['rho'] < 0
            else 'fail to reject H0'
        )

print('=== RQ3: Spearman CC vs test quality (Holm on', len(testable), 'testable pairs) ===')
for r in rq3_results:
    adj = r.get('adj_p', 'N/A')
    adj_str = f'{adj:.6g}' if isinstance(adj, float) else str(adj)
    rho_str = f'{r["rho"]:.4f}' if isinstance(r.get('rho'), float) else str(r.get('rho', 'NA'))
    raw_str = f'{r["raw_p"]:.6g}' if isinstance(r.get('raw_p'), float) else str(r.get('raw_p', 'NA'))
    print(f"  {r['label']:35s}: n={r['n']}, rho={rho_str}, raw_p={raw_str}, adj_p={adj_str} → {r.get('decision', '?')}")

In [ ]:
# Cell 9 — Sensitivity analysis: worst-case (non-executable = 0%)
THRESHOLD = 75.0

# Java worst-case: 1 non-executable → add 0.0
java_bc_wc = pd.concat([java_bc, pd.Series([0.0])], ignore_index=True)
# Python worst-case: 3 non-executable → add 3x 0.0
py_bc_wc   = pd.concat([py_bc,   pd.Series([0.0, 0.0, 0.0])], ignore_index=True)

for label, vals in [('Java (worst-case)', java_bc_wc), ('Python (worst-case)', py_bc_wc)]:
    above = (vals > THRESHOLD).sum()
    below = (vals < THRESHOLD).sum()
    n_nt  = above + below
    result = binomtest(int(above), int(n_nt), 0.5, alternative='greater') if n_nt > 0 else None
    wc_p = float(result.pvalue) if result else 1.0
    print(f"{label}: n={len(vals)}, above={above}, below={below}, p={wc_p:.6g}")

In [ ]:
# Cell 10 — Load and display final summary.csv
summary = pd.read_csv(RESULTS / 'summary.csv')
display_cols = ['rq', 'analysis_label', 'n_valid', 'median', 'raw_p_value',
                'adjusted_p_value', 'effect_estimate', 'decision']
print(summary[display_cols].to_string(index=False))

## Conclusions per RQ

### RQ1 — Branch coverage vs 75% practical target

**Java**: n=24 valid, median=95.45%, sign test k=23/24 above threshold.  
Raw p=1.49×10⁻⁶, Holm-adjusted p=2.98×10⁻⁶ < 0.05 → **reject H0**.  
GPT-4o achieves branch coverage above 75% on Java units with CC 5–15.

**Python**: n=22 valid, median=100.00%, sign test k=19/22 above threshold.  
Raw p=4.28×10⁻⁴, Holm-adjusted p=4.28×10⁻⁴ < 0.05 → **reject H0**.  
GPT-4o achieves branch coverage above 75% on Python units with CC 5–15.

**Worst-case sensitivity** (non-executable = 0%): Java p=9.72×10⁻⁶, Python p=7.32×10⁻³ — both still reject H0. Result is robust.

---

### RQ2 — GPT-4o vs Randoop mutation score (Java only)

Paired n=24, median GPT=86.61%, median Randoop=15.38%.  
Median difference=62.14 pp, Bootstrap 95% CI=[31.58, 84.62].  
Wilcoxon W=190, p=6.58×10⁻⁵ < 0.05 → **reject H0**.  
Sign-test sensitivity: pos=19, neg=0, ties=5, p=1.91×10⁻⁶.  
GPT-4o mutation score is significantly higher than Randoop on Java units with CC 5–15.

---

### RQ3 — Cyclomatic Complexity vs test quality (Spearman)

| Pair | n | ρ | Holm-adjusted p | Decision |
|---|---|---|---|---|
| Java × branch_coverage | 24 | -0.517 | 0.0146 | **reject H0** |
| Java × mutation_score | 24 | +0.109 | 0.695 | fail to reject H0 |
| Python × branch_coverage | 22 | -0.512 | 0.0148 | **reject H0** |
| Python × mutation_score | 0 | N/A | EXCLUDED | NOT_TESTABLE |

For branch coverage, higher CC is associated with lower GPT-4o test coverage (ρ≈−0.51 for both languages). Mutation score shows no significant correlation with CC in Java; Python mutation data is unavailable (pytest-mutagen generated 0 mutants across all 25 functions).

---

> **Scope limitation**: All conclusions are scoped to 50 units (25 Java + 25 Python) with CC 5–15, using first-attempt zero-shot prompt. Results do not generalise to CC outside this range, other models, or repaired tests.